In [0]:
from pyspark.sql import functions as F
S = "workspace.default"

cbz = spark.table(f"{S}.citibike_zoned")
lk = spark.table(f"{S}.taxi_zone_lookup")
pu = lk.select(F.col("zone").alias("start_zone"), F.col("borough").alias("pu_borough"))
do = lk.select(F.col("zone").alias("end_zone"), F.col("borough").alias("do_borough"))
tz = spark.table(f"{S}.traffic_zone_hourly").select(
    F.col("zone").alias("start_zone"), F.col("hour").alias("start_hour"), "is_weekday",
    F.col("avg_vol_per_hour").alias("pu_traffic"))

feat = (cbz.withColumn("is_weekday", F.col("dow").between(2,6))
        .join(pu, "start_zone").join(do, "end_zone")
        .join(tz, ["start_zone","start_hour","is_weekday"], "left")
        .withColumn("same_zone", (F.col("start_zone")==F.col("end_zone")).cast("int")))
feat.write.mode("overwrite").saveAsTable(f"{S}.bike_features")

print(cbz.count(), feat.count())
feat.select("start_zone","end_zone","pu_borough","do_borough","start_hour","is_weekday","pu_traffic","minutes").show(5)

4712721 4712721
+----------+--------+----------+----------+----------+----------+----------+------------------+
|start_zone|end_zone|pu_borough|do_borough|start_hour|is_weekday|pu_traffic|           minutes|
+----------+--------+----------+----------+----------+----------+----------+------------------+
|        36|     198|  Brooklyn|    Queens|         9|     false|      NULL|               9.3|
|        36|     225|  Brooklyn|  Brooklyn|         2|     false|      NULL|             11.95|
|       161|     229| Manhattan| Manhattan|        13|      true|      NULL|               5.9|
|       140|     163| Manhattan| Manhattan|         9|      true|      NULL|12.983333333333333|
|       170|     229| Manhattan| Manhattan|        21|     false|      NULL| 6.866666666666666|
+----------+--------+----------+----------+----------+----------+----------+------------------+
only showing top 5 rows


In [0]:
t = feat.filter(F.col("is_weekday") & F.col("start_hour").isin(16,17,18))
train = t.filter(F.col("started_at") < "2025-10-22")
test  = t.filter(F.col("started_at") >= "2025-10-22")

base = (train.groupBy("start_zone","end_zone")
        .agg(F.count("*").alias("n"),
             F.percentile_approx("minutes",0.5).alias("med"),
             F.percentile_approx("minutes",0.1).alias("p10"),
             F.percentile_approx("minutes",0.9).alias("p90"))
        .filter("n >= 20"))
base.write.mode("overwrite").saveAsTable(f"{S}.bike_baseline")

j = test.join(base, ["start_zone","end_zone"])
(j.withColumn("abs_err", F.abs(F.col("minutes")-F.col("med")))
  .withColumn("covered", ((F.col("minutes")>=F.col("p10")) & (F.col("minutes")<=F.col("p90"))).cast("int"))
  .agg(F.count("*").alias("test_trips_with_baseline"),
       F.round(F.avg("abs_err"),2).alias("MAE_min"),
       F.round(F.avg("covered"),3).alias("p10_p90_coverage"))).show()

print("test trips:", test.count(), " zone pairs in baseline:", base.count())
print("share of rides with a traffic number:", feat.filter("pu_traffic is not null").count()/feat.count())

+------------------------+-------+----------------+
|test_trips_with_baseline|MAE_min|p10_p90_coverage|
+------------------------+-------+----------------+
|                  299004|   3.28|           0.802|
+------------------------+-------+----------------+

test trips: 315672  zone pairs in baseline: 3833
share of rides with a traffic number: 0.24951784754497455


In [0]:
from pyspark.sql import functions as F
S = "workspace.default"

t = spark.table(f"{S}.trips_zoned")
lk = spark.table(f"{S}.taxi_zone_lookup")
pu = lk.select(F.col("zone").alias("pu_zone"), F.col("borough").alias("pu_b"))
do = lk.select(F.col("zone").alias("do_zone"), F.col("borough").alias("do_b"))
t = (t.filter(F.col("dow").between(2,6) & F.col("hour").isin(16,17,18))
       .join(pu, "pu_zone").join(do, "do_zone"))

train = t.filter(F.col("month") < 10)    # Jan, Apr, Jul
test  = t.filter(F.col("month") == 10)   # October

def stats(df, keys):
    return (df.groupBy("mode", *keys)
            .agg(F.count("*").alias("n"),
                 F.percentile_approx("minutes", 0.5).alias("med"),
                 F.percentile_approx("minutes", 0.1).alias("p10"),
                 F.percentile_approx("minutes", 0.9).alias("p90")))

zone = stats(train, ["pu_zone","do_zone"]).filter("n >= 20")
boro = stats(train, ["pu_b","do_b"])
zone.write.mode("overwrite").saveAsTable(f"{S}.baseline_zone")
boro.write.mode("overwrite").saveAsTable(f"{S}.baseline_borough")

z = zone.select("mode","pu_zone","do_zone", F.col("med").alias("z_med"), F.col("p10").alias("z_p10"), F.col("p90").alias("z_p90"))
b = boro.select("mode","pu_b","do_b", F.col("med").alias("b_med"), F.col("p10").alias("b_p10"), F.col("p90").alias("b_p90"))

j = (test.join(z, ["mode","pu_zone","do_zone"], "left").join(b, ["mode","pu_b","do_b"], "left")
         .withColumn("med", F.coalesce("z_med","b_med"))
         .withColumn("p10", F.coalesce("z_p10","b_p10"))
         .withColumn("p90", F.coalesce("z_p90","b_p90"))
         .withColumn("used_zone", F.col("z_med").isNotNull().cast("int"))
         .filter(F.col("med").isNotNull()))

(j.withColumn("abs_err", F.abs(F.col("minutes") - F.col("med")))
  .withColumn("covered", ((F.col("minutes") >= F.col("p10")) & (F.col("minutes") <= F.col("p90"))).cast("int"))
  .groupBy("mode")
  .agg(F.count("*").alias("test_trips"),
       F.round(F.avg("used_zone"), 3).alias("share_zone_level"),
       F.round(F.avg("abs_err"), 2).alias("MAE_min"),
       F.round(F.avg("covered"), 3).alias("p10_p90_coverage"))
  .orderBy("mode")).show()

+----+----------+----------------+-------+----------------+
|mode|test_trips|share_zone_level|MAE_min|p10_p90_coverage|
+----+----------+----------------+-------+----------------+
|bike|    978821|           0.979|   3.89|           0.809|
| fhv|     23099|           0.262|   9.14|           0.755|
|taxi|     20505|           0.685|   6.81|           0.699|
+----+----------+----------------+-------+----------------+

